# Train HARNN — Word2Vec + Global

Paper NLPIR 2023 · Word2Vec self-trained · Global approach

| Cell | Nhiệm vụ |
|------|----------|
| 1 | Import, GPU, load data |
| 2 | Dataset & DataLoader |
| 3 | Train Word2Vec → embedding matrix |
| 4 | HARNN model |
| 5 | Training loop |
| 6 | Đánh giá test set |
| 7 | learning curve |


## Cell 1 — Import, GPU, load data


In [ ]:
import json, warnings
import numpy as np
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    average_precision_score,
)

warnings.filterwarnings("ignore")

DATA_DIR = Path(r"C:\Users\Admin\Documents\nlp\NLP_Project\data\process_data")
OUTPUT_DIR = Path(r"C:\Users\Admin\Documents\nlp\NLP_Project\output")
[(OUTPUT_DIR / d).mkdir(parents=True, exist_ok=True) for d in ["models/checkpoints", "results", "figures", "log"]]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Tải dữ liệu
with open(DATA_DIR / "dataset.json", encoding="utf-8") as f:
    articles = json.load(f)
with open(DATA_DIR / "vocab.json", encoding="utf-8") as f:
    vocab = json.load(f)

VOCAB_SIZE = len(vocab)
NUM_CLASSES = [
    len(articles[0]["vec_l1"]),
    len(articles[0]["vec_l2"]),
    len(articles[0]["vec_l3"]),
]

print(f"Device: {device} | Data: {len(articles):,} | Vocab: {VOCAB_SIZE:,} | Classes: {NUM_CLASSES}")

## Cell 2 — Dataset & DataLoader


In [ ]:
import json, random, torch, numpy as np
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

MAX_LEN, BATCH_SIZE, SEED, MIN_CROP = 512, 16, 42, 20
DATA_DIR = Path(r"C:\Users\Admin\Documents\nlp\NLP_Project\data")


class VnExpressDataset(Dataset):
    def __init__(self, articles, vocab, augment=False):
        self.articles, self.vocab, self.augment = articles, vocab, augment

    def __len__(self):
        return len(self.articles)

    def __getitem__(self, idx):
        a = self.articles[idx]
        tokens = a["tokens"]

        if self.augment and len(tokens) > MIN_CROP:
            n = len(tokens)
            c = random.randint(max(MIN_CROP, n * 3 // 10), n)
            start = random.randint(0, n - c)
            tokens = tokens[start : start + c]

        ids = [self.vocab.get(t, 1) for t in tokens][:MAX_LEN]
        ids += [0] * (MAX_LEN - len(ids))

        return (
            torch.tensor(ids, dtype=torch.long),
            torch.tensor(a["vec_l1"], dtype=torch.float),
            torch.tensor(a["vec_l2"], dtype=torch.float),
            torch.tensor(a["vec_l3"], dtype=torch.float),
        )


def split_data(articles, seed=SEED):
    y = np.array([a["vec_l1"] + a["vec_l2"] + a["vec_l3"] for a in articles], dtype=np.int32)
    idx = np.arange(len(articles)).reshape(-1, 1)

    def isplit(x, y, r):
        return next(MultilabelStratifiedShuffleSplit(1, test_size=r, random_state=seed).split(x, y))

    tr, ho = isplit(idx, y, 0.2)
    vl, te = isplit(idx[ho], y[ho], 0.5)

    return (
        [articles[i] for i in tr],
        [articles[ho[i]] for i in vl],
        [articles[ho[i]] for i in te],
    )

train_arts, val_arts, test_arts = split_data(articles)

def make_loader(arts, aug, shuf):
    return DataLoader(
        VnExpressDataset(arts, vocab, aug),
        batch_size=BATCH_SIZE,
        shuffle=shuf,
        num_workers=0,
    )

train_loader = make_loader(train_arts, True, True)
val_loader = make_loader(val_arts, False, False)
test_loader = make_loader(test_arts, False, False)

DATA_DIR.mkdir(parents=True, exist_ok=True)
for name, data in [("train_data", train_arts), ("test_data", test_arts)]:
    with open(DATA_DIR / f"{name}.json", "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

print(f"Train: {len(train_arts):,} | Val: {len(val_arts):,} | Test: {len(test_arts):,}")
print(f"Batches/epoch: {len(train_loader)} | Split: Iterative Stratification")

## Cell 3 — Train Word2Vec → embedding matrix
Tham số theo paper: `vector_size=100`, `min_count=5`, `epochs=10`, Skip-gram.


In [ ]:
from gensim.models import Word2Vec

EMBED_DIM = 100
corpus = [a["tokens"] for a in articles]

# 1. Huấn luyện và lưu Word2Vec
w2v = Word2Vec(corpus, vector_size=EMBED_DIM, min_count=5, epochs=10, sg=1, workers=4, seed=SEED)
w2v_path = OUTPUT_DIR / "models" / "checkpoints" / "word2vec.model"
# Tạo thư mục nếu chưa có
w2v_path.parent.mkdir(parents=True, exist_ok=True)
w2v.save(str(w2v_path))

# 2. Khởi tạo và nạp Embedding Matrix
embed_matrix = np.zeros((VOCAB_SIZE, EMBED_DIM), dtype=np.float32)
embed_matrix[1] = w2v.wv.vectors.mean(axis=0)  # Vector trung bình cho thẻ UNK

found = 0
for w, i in vocab.items():
    if w in w2v.wv:
        embed_matrix[i] = w2v.wv[w]
        found += 1

print(f"W2V Vocab: {len(w2v.wv):,} từ | Coverage: {found/(VOCAB_SIZE-2):.2%}")
if "kinh_tế" in w2v.wv:
    print(f"Từ gần 'kinh_tế': {[w for w, _ in w2v.wv.most_similar('kinh_tế', topn=5)]}")

## Cell 4 — HARNN model
`DRL` (Embedding + BiGRU) → `HARL` (Attention × 3) → `HAM` (LSTMCell) → `HPL` (Linear + Sigmoid)


In [ ]:
import torch
import torch.nn as nn

class HARNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, num_classes, pretrained_embeddings=None, dropout=0.5):
        super().__init__()
        self.num_levels = len(num_classes)
        self.hidden_size = hidden_size

        if pretrained_embeddings is not None:
            self.emb = nn.Embedding.from_pretrained(torch.from_numpy(pretrained_embeddings), freeze=False, padding_idx=0)
        else:
            self.emb = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
            
        self.bigru = nn.GRU(embed_dim, hidden_size, bidirectional=True, batch_first=True)
        self.drop = nn.Dropout(dropout)

        # HARL & HAM & HPL
        self.attention = nn.ModuleList([nn.Linear(hidden_size * 2, 1) for _ in range(self.num_levels)])
        self.ham = nn.LSTMCell(hidden_size * 2, hidden_size)
        self.classifiers = nn.ModuleList([nn.Linear(hidden_size * 3, n) for n in num_classes])

    def forward(self, x):
        doc = self.drop(self.bigru(self.drop(self.emb(x)))[0])
        
        h, c = (torch.zeros(x.size(0), self.hidden_size, device=x.device) for _ in range(2))
        
        preds = []
        for i in range(self.num_levels):
            ctx = (torch.softmax(self.attention[i](doc), dim=1) * doc).sum(1)
            h, c = self.ham(ctx, (h, c))
            feat = self.drop(torch.cat([ctx, h], dim=-1))
            
            preds.append(self.classifiers[i](feat))
            
        return preds

HIDDEN_SIZE = 256
model = HARNN(VOCAB_SIZE, EMBED_DIM, HIDDEN_SIZE, NUM_CLASSES, embed_matrix).to(device)

print(f"Params: {sum(p.numel() for p in model.parameters()):,}")
dummy_in = torch.zeros(2, MAX_LEN, dtype=torch.long, device=device)
print(f"Output shapes: {[tuple(o.shape) for o in model(dummy_in)]}")

## Cell 5 — Training loop


In [ ]:
import time, json, torch
import torch.nn as nn, numpy as np
from sklearn.metrics import f1_score

NUM_EPOCHS, LR, PATIENCE = 10, 3e-4, 5
LEVEL_WEIGHTS = [1.0, 1.0, 1.2]
CKPT_PATH = OUTPUT_DIR / "models" / "checkpoints" / "best_model.pt"

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", patience=2, factor=0.5)

criterion = nn.BCEWithLogitsLoss()

def run_epoch(loader, training=True):
    model.train(training)
    total_loss = 0
    all_preds, all_labels = [[] for _ in range(3)], [[] for _ in range(3)]

    with torch.set_grad_enabled(training):
        for ids, l1, l2, l3 in loader:
            ids, targets = ids.to(device), [t.to(device) for t in (l1, l2, l3)]
            if training: optimizer.zero_grad()

            preds = model(ids)
            losses = [LEVEL_WEIGHTS[i] * criterion(preds[i], targets[i]) for i in range(3) if targets[i].sum() > 0]
            
            if losses:
                loss = sum(losses)
                if training:
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                    optimizer.step()
                total_loss += loss.item()

            for i in range(3):
                probs = torch.sigmoid(preds[i]).detach().cpu().numpy()
                all_preds[i].append(probs)
                all_labels[i].append(targets[i].cpu().numpy())

    f1s = [f1_score(np.concatenate(all_labels[i]), (np.concatenate(all_preds[i]) >= 0.5).astype(int), 
                    average="micro", zero_division=0) if sum(len(x) for x in all_labels[i]) else 0.0 for i in range(3)]
    return total_loss / len(loader), f1s

history = {k: [] for k in ["train_loss", "val_loss", "val_f1_l1", "val_f1_l2", "val_f1_l3"]}
best_f1, no_improve = 0.0, 0

print("Training started...")

for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, _      = run_epoch(train_loader, True)
    val_loss, v_f1s = run_epoch(val_loader, False)

    for k, v in zip(history.keys(), [tr_loss, val_loss, *v_f1s]): history[k].append(v)

    scheduler.step(v_f1s[0])
    global_f1 = sum(v_f1s) / 3
    
    status = ""
    if global_f1 > best_f1:
        best_f1 = global_f1
        torch.save({"epoch": epoch, "model_state": model.state_dict(), "val_f1_l1": v_f1s[0], "val_f1s": v_f1s, "global_f1": global_f1}, CKPT_PATH)
        status = " [Saved]"
        no_improve = 0 
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            status = " [Early Stop]"

    print(f"Ep {epoch}/{NUM_EPOCHS} | Train: {tr_loss:.4f} | Val: {val_loss:.4f} | F1: {v_f1s[0]:.3f} (L1), {v_f1s[1]:.3f} (L2), {v_f1s[2]:.3f} (L3){status}")
    
    if "Stop" in status:
        break

with open(OUTPUT_DIR / "results" / "train_history.json", "w") as f: json.dump(history, f, indent=2)
print(f"\nBest Global F1: {best_f1:.3f}")

## Cell 6 — Đánh giá test set


In [ ]:
import numpy as np, json, torch
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score

# 1. Load Model
model.load_state_dict(torch.load(CKPT_PATH, map_location=device)["model_state"])
model.eval()

# 2. Predict (Ép Logit qua Sigmoid)
@torch.no_grad()
def evaluate(loader):
    preds, labels = [[] for _ in range(3)], [[] for _ in range(3)]
    for ids, *targs in loader:
        out = model(ids.to(device))
        for i in range(3):
            preds[i].append(torch.sigmoid(out[i]).cpu().numpy())
            labels[i].append(targs[i].numpy())
    return [np.concatenate(p) for p in preds], [np.concatenate(l) for l in labels]

# 3. Tính Metrics (Lọc index 1 lần để tối ưu)
def get_metrics(p, l, th=0.3):
    v = l.sum(axis=1) > 0
    if not v.any(): return dict(precision=0, recall=0, f1=0, auprc=0)
    pv, lv = p[v], l[v]
    pb = (pv >= th).astype(int)
    return {
        "precision": precision_score(lv, pb, average="micro", zero_division=0),
        "recall": recall_score(lv, pb, average="micro", zero_division=0),
        "f1": f1_score(lv, pb, average="micro", zero_division=0),
        "auprc": average_precision_score(lv, pv, average="micro")
    }

p_list, l_list = evaluate(test_loader)
res = {name: get_metrics(p_list[i], l_list[i]) for i, name in enumerate(["L1", "L2", "L3"])}
res["Global"] = {k: float(np.mean([res[n][k] for n in ["L1", "L2", "L3"]])) for k in ["precision", "recall", "f1"]}

# 4. In bảng kết quả và Lưu file
print(f"{'Level':<8} | {'Precision':>9} | {'Recall':>9} | {'F1':>9} | {'AUPRC':>9}\n" + "-"*55)
for n in ["L1", "L2", "L3"]:
    m = res[n]
    print(f"{n:<8} | {m['precision']:>9.3f} | {m['recall']:>9.3f} | {m['f1']:>9.3f} | {m['auprc']:>9.3f}")
g = res["Global"]
print(f"{'-'*55}\n{'Global':<8} | {g['precision']:>9.3f} | {g['recall']:>9.3f} | {g['f1']:>9.3f}\n")

with open(OUTPUT_DIR / "results" / "results_w2v_global.json", "w") as f: 
    json.dump(res, f, indent=2)

## Cell 7 — learning curve


In [ ]:
# Learning curve (rút gọn)
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

history = json.load(open(OUTPUT_DIR / "results" / "train_history.json"))
epochs = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    "Learning Curve — HARNN Word2Vec + Global", fontsize=13, fontweight="bold", y=1.01
)

# ── Loss
ax = axes[0]
for key, color, label in [
    ("train_loss", "#2E86AB", "Train loss"),
    ("val_loss", "#E84855", "Val loss"),
]:
    ax.plot(epochs, history[key], color=color, linewidth=2, label=label)
ax.set_title("Loss", fontsize=12, fontweight="bold")
ax.set_xlabel("Epoch", fontsize=10)
ax.set_ylabel("BCE Loss", fontsize=10)

# ── Val F1
ax = axes[1]
for key, color, label in [
    ("val_f1_l1", "#2E86AB", "F1 L1"),
    ("val_f1_l2", "#F18F01", "F1 L2"),
    ("val_f1_l3", "#3BB273", "F1 L3"),
]:
    ax.plot(epochs, history[key], color=color, linewidth=2, label=label)
ax.set_title("Val F1", fontsize=12, fontweight="bold")
ax.set_xlabel("Epoch", fontsize=10)
ax.set_ylabel("F1-score", fontsize=10)
ax.set_ylim(0, 1.05)

# ── Style chung
for ax in axes:
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    ax.legend(
        fontsize=9, framealpha=0.8, loc="lower right" if ax is axes[1] else "best"
    )
    ax.grid(True, linestyle="--", alpha=0.4)
    ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
fig_path = OUTPUT_DIR / "figures" / "learning_curve.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")